# Advanced RAG Systems #1 — Hybrid Search + Reranking

Naive vector search vs the 2026 production playbook, on the same tiny knowledge base.

**The question:** *What is the SLA (Service Level Agreement) refund percentage for enterprise downtime exceeding 4 hours?*

Run the cells top to bottom (Runtime → Run all).

In [1]:
!pip install -q sentence-transformers rank-bm25

## The corpus — 11 chunks, with traps

Three plans with three different percentages (10% / 15% / 25%), an FAQ that talks *about* refunds but contains no number, and an overview page full of the right words with no answer.

In [ ]:
CORPUS = [
    ("sla_standard",
     "Nimbus Cloud SLA: Standard plan customers receive a 10% service credit when monthly "
     "downtime exceeds 1 hour. Credits are applied automatically to the next invoice. "
     "The Standard plan SLA covers compute and storage services only."),
    ("sla_enterprise",
     "Nimbus Cloud SLA: Enterprise tier customers receive a 25% service credit when monthly "
     "downtime exceeds 4 hours. The Enterprise SLA covers compute, storage, and networking. "
     "Claims must be filed within 30 days of the incident."),
    ("enterprise_support",
     "Enterprise support includes 24/7 phone support with a 15-minute response time for "
     "critical issues. Enterprise customers get a dedicated account manager and quarterly "
     "business reviews."),
    ("refund_policy",
     "Refund policy: monthly plans can be refunded within 7 days of purchase, annual plans "
     "within 30 days. Enterprise contracts are non-refundable except as stated in the SLA."),
    ("sla_pro",
     "Nimbus Cloud SLA: Pro plan customers receive a 15% service credit when monthly "
     "downtime exceeds 2 hours. Credits are applied automatically to the next invoice."),
    ("downtime_definition",
     "Downtime is measured as minutes when the service returns 5xx errors for over 60 "
     "consecutive seconds, excluding scheduled maintenance windows announced 72 hours in advance."),
    ("status_page",
     "Check status.nimbus.example for real-time availability. Historical uptime for the last "
     "90 days is published there with per-region breakdowns."),
    ("credits_process",
     "Service credits are applied to the next invoice automatically. Credits cannot be "
     "converted to cash refunds under any plan."),
    ("sla_overview",
     "Nimbus Cloud's SLA program reflects our commitment to reliability. Every plan includes "
     "service level objectives for uptime, measured monthly. Enterprise customers enjoy "
     "enhanced SLA terms with higher service credits for extended outages. See the "
     "plan-specific pages for exact percentages."),
    ("outage_postmortem",
     "Postmortem: the March 3 enterprise outage lasted 4.5 hours and SLA credits were issued "
     "per policy. Root cause was a load balancer misconfiguration. No action needed."),
    ("sla_faq",
     "SLA FAQ: what is an SLA refund? Service credits compensate customers for downtime. "
     "How do I claim? File a claim within 30 days. Which plans are covered? All paid plans "
     "include an SLA."),
]

QUERY = "What is the SLA refund percentage for enterprise downtime exceeding 4 hours?"
CORRECT_ID = "sla_enterprise"



## Step 1 — Naive retrieval: embed, cosine, top-3

One embedding model turns every chunk and the question into vectors. Cosine similarity ranks the chunks. Watch which chunk wins — and whether it contains the answer.

In [ ]:
from sentence_transformers import SentenceTransformer
import numpy as np

emb = SentenceTransformer("all-MiniLM-L6-v2")
texts = [t for _, t in CORPUS]
E = emb.encode(texts, normalize_embeddings=True)
q = emb.encode([QUERY], normalize_embeddings=True)[0]

cos = E @ q                      # cosine similarity of query vs every chunk
naive = list(np.argsort(-cos))[:3]

for r, i in enumerate(naive):
    cid, _ = CORPUS[i]
    mark = "  <-- CORRECT (25%)" if cid == CORRECT_ID else ""
    print(f"#{r+1} {cid:20s} score={cos[i]:.4f}{mark}")


**The failure:** the FAQ wins on meaning-similarity (it repeats the question's own words) while containing no percentage. The correct chunk sits at #2. Retrieval quality is the ceiling — the LLM can only stand on what it is handed.

## Step 2 — Advanced: hybrid + fusion + rerank

1. **BM25** (Best Matching 25) — keyword search: exact words, rare words weighted more.
2. **Dense** — the same vector search as Step 1.
3. **RRF** (Reciprocal Rank Fusion) — merge the two ranked lists: a chunk scores `1/(60 + its rank)` in each list, summed.
4. **Cross-encoder rerank** — a small model reads the question *together with* each candidate chunk and re-scores. A question asking 'what is the percentage' needs a chunk containing a percentage.

In [ ]:
from rank_bm25 import BM25Okapi
from sentence_transformers import CrossEncoder

def rrf(rank_lists, k=60):
    scores = {}
    for ranks in rank_lists:
        for rank, idx in enumerate(ranks):
            scores[idx] = scores.get(idx, 0) + 1.0 / (k + rank + 1)
    return sorted(scores, key=scores.get, reverse=True)

tok = [t.lower().split() for t in texts]
bm25 = BM25Okapi(tok)
b = bm25.get_scores(QUERY.lower().split())
top_bm25 = list(np.argsort(-b))[:8]
top_dense = list(np.argsort(-cos))[:8]
fused = rrf([top_bm25, top_dense])[:8]

reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
rs = reranker.predict([(QUERY, texts[i]) for i in fused])
reranked = [fused[i] for i in np.argsort(-rs)]

for r, i in enumerate(reranked[:3]):
    cid, _ = CORPUS[i]
    mark = "  <-- CORRECT (25%)" if cid == CORRECT_ID else ""
    print(f"#{r+1} {cid:20s} rerank score={rs[list(fused).index(i)]:.4f}{mark}")

print("\nTop chunk handed to the LLM:")
print(CORPUS[reranked[0]][1])


## Takeaway

Same corpus, same question — the retrieval stack was the entire difference. Naive RAG (chunk → embed → top-k) is the floor; production RAG is a portfolio of techniques, each fixing one failure mode of the last.

*Inside the LLM — Advanced RAG Systems #1*